# 03 - Targets, thresholds and sequence geometry

**Targets** - percentage return from the current close:

| horizon | candles | formula |
|---|---|---|
| 15m | 1 | `100*(close[t+1]/close[t]-1)` |
| 30m | 2 | `100*(close[t+2]/close[t]-1)` |
| 45m | 3 | `100*(close[t+3]/close[t]-1)` |
| 60m | 4 | `100*(close[t+4]/close[t]-1)` |
| 75m | 5 | `100*(close[t+5]/close[t]-1)` |
| 90m | 6 | `100*(close[t+6]/close[t]-1)` |

These are the only places future data is touched. A target that would span a
data gap is set to NaN rather than silently measuring across missing candles.

**Thresholds** are derived from the **training split only** and then frozen.

A single fixed bps threshold would leave 15m almost entirely NEUTRAL and 90m
almost entirely directional, because return dispersion grows with the square
root of the horizon. The default mode therefore places the NEUTRAL band at the
central 40% of the *train* return distribution, giving roughly 30/40/30 at every
horizon. Configure via `THRESHOLD_MODE` and `NEUTRAL_QUANTILE` in `src/config.py`.

In [ ]:
import sys, os, json
from pathlib import Path

# Make src/ importable whether this notebook runs from notebooks/ or the root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

In [ ]:
import pandas as pd

from preprocessing import preprocess
from features import build_features, FEATURE_COLUMNS
from targets import add_future_returns, return_distribution_report
from config import HORIZONS, HORIZON_LABELS, FUTURE_RETURN_COLUMNS

clean, _ = preprocess()
labelled = add_future_returns(build_features(clean))

dist = return_distribution_report(labelled)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
dist[["count", "mean_%", "std_%", "skew", "p01_%", "p05_%", "p25_%",
      "p50_%", "p75_%", "p95_%", "p99_%", "pct_positive"]]

In [ ]:
# The target distributions are what justify the thresholding rule.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, h in zip(axes.ravel(), HORIZONS):
    s = labelled[f"future_return_{HORIZON_LABELS[h]}"].dropna()
    ax.hist(s, bins=200, range=(s.quantile(0.001), s.quantile(0.999)))
    ax.set_title(f"{HORIZON_LABELS[h]}  (std={s.std():.3f}%)")
    ax.set_xlabel("future return (%)")
plt.tight_layout()
plt.show()

print("std by horizon (grows ~sqrt(horizon), the random-walk signature):")
for h in HORIZONS:
    print(f"  {HORIZON_LABELS[h]:>4}: "
          f"{labelled[f'future_return_{HORIZON_LABELS[h]}'].dropna().std():.4f}%")

In [ ]:
# Thresholds - exactly as computed on the TRAIN split by src/pipeline.py step 7.
import json
import numpy as np

from targets import compute_thresholds, thresholds_to_jsonable
from config import LOOKBACK, TRAIN_FRACTION, PURGE_SAMPLES, HORIZONS

work = labelled.dropna(subset=FUTURE_RETURN_COLUMNS).iloc[500:].reset_index(drop=True)
y = work[FUTURE_RETURN_COLUMNS].to_numpy(dtype=float)

# y_*.npy holds ONE ROW PER SAMPLE, and a sample's target is the target of the
# LAST candle in its window. So the training targets are the rows that belong
# to a training window's final candle - offset by LOOKBACK-1, never from row 0.
n_samples = len(y) - LOOKBACK + 1
n_train = int(n_samples * TRAIN_FRACTION)
train_targets = y[LOOKBACK - 1: LOOKBACK - 1 + n_train]

thr = compute_thresholds(train_targets, mode="train_quantile")
print(json.dumps(thresholds_to_jsonable(thr), indent=2))


In [ ]:
# Sequence geometry: 64 candles in, 6 horizons out.
import numpy as np

from sequences import make_sequences, valid_window_starts

X2d = work[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
starts = valid_window_starts(
    work["segment_id"].to_numpy(),
    work[FUTURE_RETURN_COLUMNS].notna().all(axis=1).to_numpy(),
    LOOKBACK,
)
print("2D matrix      :", X2d.shape, f"({X2d.nbytes / 1e6:.1f} MB)")
print("candidate windows:", f"{len(X2d) - LOOKBACK + 1:,}")
print("valid windows  :", f"{starts.size:,}  "
      f"({len(X2d) - LOOKBACK + 1 - starts.size:,} excluded: gap or missing target)")

w = make_sequences(X2d, LOOKBACK)          # a VIEW - costs no memory
print("3D view (all)  :", w.shape,
      f"-> {np.prod(w.shape) * 4 / 1e9:.2f} GB if fully materialised")

# Always index the view THROUGH `starts`. Window k is w[starts[k]], never w[k]:
# excluded windows leave holes in `starts`, so w[k] may be a different window
# (or one that straddles a data gap).
batch = np.ascontiguousarray(w[starts[:512]])
print("one batch      :", batch.shape, f"({batch.nbytes / 1e6:.1f} MB)")
print("\nSample k uses rows [starts[k], starts[k]+63]; its target is the target "
      "of row starts[k]+63.")


In [ ]:
# The canonical split. src/pipeline.py applies exactly this to the real data.
from sequences import chronological_split
from config import PURGE_SAMPLES

sp = chronological_split(starts.size, train_fraction=0.70, val_fraction=0.15,
                         test_fraction=0.15, purge=PURGE_SAMPLES)
print(sp.sizes())
print(f"\npurge = {PURGE_SAMPLES} samples = LOOKBACK + max(HORIZONS) "
      f"= {LOOKBACK} + {max(HORIZONS)}")
print("Why that large: the last train sample's label is resolved from candles")
print(f"  [start + {LOOKBACK - 1}, start + {LOOKBACK - 1 + max(HORIZONS)}]")
print("so the gap to the first validation feature window must exceed")
print(f"  LOOKBACK - 1 + max(HORIZONS) = {LOOKBACK - 1 + max(HORIZONS)} rows.")
print("\nTo build the real dataset:  python src/pipeline.py")
